# Geneformer BCL6 in silico deletion on post-cardiomyocytes

Adapted from the official Geneformer in-silico perturbation tutorial:
https://huggingface.co/ctheodoris/Geneformer/blob/main/examples/in_silico_perturbation.ipynb

Geneformer's native artefact is per-cell embedding shifts, not gene-level
expression deltas. The output of this notebook is a single `.npz` with the
original and perturbed embeddings; the project's
`scripts/09_run_geneformer_bcl6.py` projects them back into expression space
via ridge regression so the downstream evaluator can score Geneformer the
same way it scores CellOracle / scGPT.

Required inputs that have to be produced outside the notebook (one-time
setup, cells with comments below):
  - Loom or h5ad file with raw counts and an `ensembl_id` column in var.
  - Tokenized HuggingFace dataset (Geneformer's `TranscriptomeTokenizer`).
  - Pretrained Geneformer model directory (downloaded from Hugging Face).

If you already have Geneformer embeddings (original + perturbed) in any
format, you can skip this notebook entirely. Save them as a `.npz` with three
arrays — `cell_ids`, `original_embeddings`, `perturbed_embeddings` — and run
`scripts/09_run_geneformer_bcl6.py` directly.

Environment: `envs/geneformer.yml`.

In [ ]:
import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import yaml

In [ ]:
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

with open('configs/config.yaml') as f:
    config = yaml.safe_load(f)

EXPR_PATH = Path(config['paths']['exported_expression_csv'])
META_PATH = Path(config['paths']['exported_meta_csv'])
NATIVE_OUT = Path(config['model_runners']['geneformer']['native_output_path'])
TARGET_GENE_CANDIDATES = config['perturbation']['target_gene_candidates']

# Geneformer pretrained model directory — clone or download from Hugging Face:
#   git lfs install
#   git clone https://huggingface.co/ctheodoris/Geneformer
GENEFORMER_DIR = Path(os.environ.get('GENEFORMER_DIR', 'data/raw/Geneformer'))
MODEL_DIR = GENEFORMER_DIR  # the repo doubles as the model directory for the v1 model
TOKENIZED_DS_PATH = Path('data/processed/perturbations/cardiomyocyte.tokenized.dataset')
INTERMEDIATE_LOOM = Path('data/processed/perturbations/cardiomyocyte.loom')
ENSEMBL_MAP_CSV = Path(os.environ.get('ENSEMBL_MAP_CSV', 'data/processed/symbol_to_ensembl.csv'))

NATIVE_OUT.parent.mkdir(parents=True, exist_ok=True)
print('Geneformer model dir:', GENEFORMER_DIR)
print('Native output target:', NATIVE_OUT)

## 1. Build AnnData with Ensembl IDs

Geneformer expects gene Ensembl IDs in `var['ensembl_id']`. The exported
Seurat matrix uses gene symbols, so we map symbols to Ensembl IDs using
mygene. The resulting symbol -> Ensembl table is cached at
`ENSEMBL_MAP_CSV` so reruns are fast.

In [ ]:
expr = pd.read_csv(EXPR_PATH, index_col=0)
expr.index = expr.index.astype(str)
expr.columns = expr.columns.astype(str)
meta = pd.read_csv(META_PATH, index_col=0)
if 'cell_id' in meta.columns:
    meta.index = meta['cell_id'].astype(str)
common = meta.index.intersection(expr.columns)
expr = expr.loc[:, common]
meta = meta.loc[common]
print('Expression shape (genes x cells):', expr.shape)

In [ ]:
if ENSEMBL_MAP_CSV.exists():
    sym2ens = pd.read_csv(ENSEMBL_MAP_CSV)
else:
    import mygene
    mg = mygene.MyGeneInfo()
    res = mg.querymany(
        expr.index.tolist(),
        scopes='symbol',
        fields='ensembl.gene',
        species='human',
        as_dataframe=True,
        verbose=False,
    )
    res = res.reset_index().rename(columns={'query': 'symbol'})
    if 'ensembl' in res.columns:
        res['ensembl_id'] = res['ensembl'].apply(
            lambda v: v[0]['gene'] if isinstance(v, list) else (v.get('gene') if isinstance(v, dict) else None)
        )
    elif 'ensembl.gene' in res.columns:
        res['ensembl_id'] = res['ensembl.gene']
    sym2ens = res[['symbol', 'ensembl_id']].dropna().drop_duplicates(subset='symbol')
    ENSEMBL_MAP_CSV.parent.mkdir(parents=True, exist_ok=True)
    sym2ens.to_csv(ENSEMBL_MAP_CSV, index=False)
    print('Wrote symbol -> Ensembl map to', ENSEMBL_MAP_CSV)

sym2ens = sym2ens.dropna().drop_duplicates(subset='symbol').set_index('symbol')
print('Symbols with Ensembl mapping:', len(sym2ens))

In [ ]:
keep_genes = [g for g in expr.index if g in sym2ens.index]
expr = expr.loc[keep_genes]
ensembl_ids = sym2ens.loc[keep_genes, 'ensembl_id'].astype(str).values

# Geneformer expects integer counts. Our exported matrix is SCT-scaled, so
# rescale to non-negative integer-ish counts via a simple monotonic mapping.
counts = np.clip(expr.values, 0, None)
counts = np.round(counts * 100).astype(np.int32)

adata = ad.AnnData(
    X=counts.T,
    obs=meta.copy(),
    var=pd.DataFrame({'ensembl_id': ensembl_ids, 'gene_symbol': keep_genes}, index=keep_genes),
)
adata.obs['n_counts'] = adata.X.sum(axis=1)
adata.obs['cell_id'] = adata.obs_names
print(adata)

## 2. Tokenize for Geneformer

Geneformer's `TranscriptomeTokenizer` consumes loom files. We dump the AnnData
to loom, then run the tokenizer.

In [ ]:
import loompy
if not INTERMEDIATE_LOOM.exists():
    INTERMEDIATE_LOOM.parent.mkdir(parents=True, exist_ok=True)
    row_attrs = {'ensembl_id': adata.var['ensembl_id'].astype(str).values}
    col_attrs = {
        'cell_id': adata.obs_names.astype(str).values,
        'n_counts': adata.obs['n_counts'].astype(int).values,
    }
    loompy.create(str(INTERMEDIATE_LOOM), adata.X.T.astype(np.int32), row_attrs, col_attrs)
    print('Wrote loom file:', INTERMEDIATE_LOOM)

from geneformer import TranscriptomeTokenizer
if not TOKENIZED_DS_PATH.exists():
    tk = TranscriptomeTokenizer(
        custom_attr_name_dict={'cell_id': 'cell_id'},
        nproc=4,
    )
    tk.tokenize_data(
        data_directory=str(INTERMEDIATE_LOOM.parent),
        output_directory=str(TOKENIZED_DS_PATH.parent),
        output_prefix=TOKENIZED_DS_PATH.stem.replace('.tokenized', '').replace('.dataset', '') + '.tokenized',
        file_format='loom',
    )
    print('Tokenization done.')
else:
    print('Reusing existing tokenized dataset at', TOKENIZED_DS_PATH)

## 3. Extract original cell embeddings

In [ ]:
from geneformer import EmbExtractor

EMB_OUT_DIR = NATIVE_OUT.parent / 'geneformer_emb'
EMB_OUT_DIR.mkdir(parents=True, exist_ok=True)

embex = EmbExtractor(
    model_type='Pretrained',
    num_classes=0,
    filter_data=None,
    max_ncells=None,
    emb_layer=-1,
    emb_label=['cell_id'],
    labels_to_plot=[],
    forward_batch_size=16,
    nproc=4,
)
embs_df = embex.extract_embs(
    model_directory=str(MODEL_DIR),
    input_data_file=str(TOKENIZED_DS_PATH),
    output_directory=str(EMB_OUT_DIR),
    output_prefix='cardiomyocyte_original',
)
print('Original embeddings shape:', embs_df.shape)

## 4. In silico BCL6 deletion → perturbed embeddings

In [ ]:
# Look up BCL6's Ensembl ID from our mapping table.
target_gene = next((g for g in TARGET_GENE_CANDIDATES if g in sym2ens.index), None)
if target_gene is None:
    raise ValueError(f'None of {TARGET_GENE_CANDIDATES} present in symbol -> Ensembl table.')
target_ensembl = sym2ens.loc[target_gene, 'ensembl_id']
print(f'BCL6 mapped to {target_ensembl}')

from geneformer import InSilicoPerturber

PERT_OUT_DIR = NATIVE_OUT.parent / 'geneformer_pert'
PERT_OUT_DIR.mkdir(parents=True, exist_ok=True)

isp = InSilicoPerturber(
    perturb_type='delete',
    perturb_rank_shift=None,
    genes_to_perturb=[target_ensembl],
    combos=0,
    anchor_gene=None,
    model_type='Pretrained',
    num_classes=0,
    emb_mode='cell',
    cell_emb_style='mean_pool',
    filter_data=None,
    cell_states_to_model=None,
    state_embs_dict=None,
    max_ncells=None,
    emb_layer=-1,
    forward_batch_size=16,
    nproc=4,
)
isp.perturb_data(
    model_directory=str(MODEL_DIR),
    input_data_file=str(TOKENIZED_DS_PATH),
    output_directory=str(PERT_OUT_DIR),
    output_prefix='cardiomyocyte_bcl6_delete',
)
print('Perturbation results saved under', PERT_OUT_DIR)

## 5. Pack original + perturbed embeddings into the project's native .npz

`InSilicoPerturber` emits per-cell perturbed embeddings as pickled dicts in
`PERT_OUT_DIR`. Below we walk those files and align cells with the original
embeddings extracted in step 3, then save a single `.npz` with three arrays
that `scripts/09_run_geneformer_bcl6.py` knows how to read.

In [ ]:
import glob

# Original embeddings DataFrame indexed by cell_id
orig = embs_df.set_index('cell_id') if 'cell_id' in embs_df.columns else embs_df
orig.index = orig.index.astype(str)

# Walk perturbation pickle outputs
pert_pickles = sorted(glob.glob(str(PERT_OUT_DIR / '*perturbed*.pickle')))
if not pert_pickles:
    pert_pickles = sorted(glob.glob(str(PERT_OUT_DIR / '*.pickle')))
if not pert_pickles:
    raise FileNotFoundError(f'No InSilicoPerturber output pickles found in {PERT_OUT_DIR}.')

perturbed_records = []
for p in pert_pickles:
    with open(p, 'rb') as f:
        rec = pickle.load(f)
    if isinstance(rec, dict):
        for key, value in rec.items():
            cell_id = key if isinstance(key, str) else getattr(key, 'cell_id', None)
            emb = np.asarray(value).reshape(-1)
            if cell_id is not None:
                perturbed_records.append((str(cell_id), emb))
    elif isinstance(rec, pd.DataFrame):
        cell_id_col = 'cell_id' if 'cell_id' in rec.columns else rec.columns[0]
        emb_cols = [c for c in rec.columns if c != cell_id_col]
        for _, row in rec.iterrows():
            perturbed_records.append((str(row[cell_id_col]), row[emb_cols].values.astype(float)))

if not perturbed_records:
    raise RuntimeError(
        'Could not parse Geneformer perturbation output. Inspect the files in '
        f'{PERT_OUT_DIR} and adapt this cell to the structure your Geneformer '
        'version writes.'
    )

pert_df = pd.DataFrame.from_records(
    [(cid, *emb) for cid, emb in perturbed_records],
    columns=['cell_id'] + [f'emb_{i}' for i in range(len(perturbed_records[0][1]))],
).set_index('cell_id')
common_cells = orig.index.intersection(pert_df.index)
if len(common_cells) == 0:
    raise ValueError('No overlap between original and perturbed cell IDs.')

orig_aligned = orig.loc[common_cells].astype(float).values
pert_aligned = pert_df.loc[common_cells].astype(float).values
print('Aligned matrices:', orig_aligned.shape, pert_aligned.shape)

In [ ]:
np.savez(
    NATIVE_OUT,
    cell_ids=np.asarray(common_cells, dtype=object),
    original_embeddings=orig_aligned.astype(np.float32),
    perturbed_embeddings=pert_aligned.astype(np.float32),
    target_gene=np.asarray([target_gene], dtype=object),
    target_ensembl=np.asarray([target_ensembl], dtype=object),
)
print('Wrote native Geneformer artefact to', NATIVE_OUT)
print('Now run: python scripts/09_run_geneformer_bcl6.py --config configs/config.yaml')